# 06 — Convolutional Neural Networks (CNNs)

CNNs use convolution + pooling layers to learn spatial hierarchies, standard for image tasks.

## Conv2d Mechanics

In [ ]:
import torch
import torch.nn as nn

conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
x = torch.randn(8, 3, 32, 32)   # (batch, channels, height, width)
out = conv(x)
out.shape   # (8, 16, 32, 32) — padding=1 keeps spatial size unchanged for kernel_size=3

# output size formula: out = floor((in + 2*padding - kernel_size) / stride) + 1


## Pooling

In [ ]:
pool = nn.MaxPool2d(kernel_size=2, stride=2)
pool(out).shape    # (8, 16, 16, 16) — halves spatial dims

avgpool = nn.AdaptiveAvgPool2d((1, 1))   # collapse to 1x1 regardless of input size, common before FC layer
avgpool(out).shape  # (8, 16, 1, 1)


## A Basic CNN (LeNet-style)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32x32 -> 16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16x16 -> 8x8
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 8x8 -> 4x4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SimpleCNN()
model(torch.randn(8, 3, 32, 32)).shape   # (8, 10)


## Modern Block: Conv + BatchNorm + ReLU

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),  # bias unneeded before BatchNorm
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


## Residual Block (ResNet-style)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + identity)   # skip connection

block = ResBlock(64)
block(torch.randn(4, 64, 16, 16)).shape


## Training on CIFAR-10 (end-to-end)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_ds = datasets.CIFAR10("data", train=True, download=True, transform=transform)
test_ds = datasets.CIFAR10("data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    print(f"epoch {epoch+1} done, last batch loss={loss.item():.4f}")


## Visualizing Feature Maps

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    x = next(iter(test_loader))[0][:1].to(device)
    feat = model.features[:3](x)   # output after first conv block

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(feat[0, i].cpu().numpy(), cmap="viridis")
    ax.axis("off")
plt.show()


## Common Gotchas
- Flatten dimensions must match: track spatial size through each conv/pool layer or use `AdaptiveAvgPool2d` to avoid hardcoding.
- Forgetting to normalize images the same way at train and inference time skews predictions.
- `bias=False` in `Conv2d` is standard when followed by `BatchNorm2d` (the BN bias makes conv bias redundant).
- Data augmentation (flips, crops) should apply only to the training transform, not validation/test.